# 02 - Build the multi-city weekly feature table

Join every feature source onto the `(adm3_pcode, date)` weekly grid emitted by `01-prepare-target.ipynb`. Output:

* `all_cities_weekly_features.csv` -- one row per `(adm3_pcode, date)`, one column per feature.

**Granularity handling** (time alignment):
* **Daily ADM4** (climate_air_quality, climate_atmosphere, climate_atmosphere_downscaled, climate_land): daily -> Monday-start weekly mean per ADM4 -> ADM3 weekly aggregate.
* **Monthly ADM4** (climate_indices): ADM3 monthly aggregate -> broadcast to all weeks in that month.
* **Annual ADM4** (worldpop, nightlights, ookla, RWI, OSM POI): ADM3 annual aggregate -> broadcast to all weeks in that year.
* **Static ADM4** (esa_worldcover, project_noah_hazards, buildings, brgy_geography, mapbox_brgy_iso, geoportal_doh_poi): ADM3 aggregate -> broadcast to all weeks.
* **Already ADM3** (mapbox_health_facility_city_isochrones): broadcast directly.

**Spatial aggregation (ADM4 -> ADM3)** depends on the *physical nature* of each variable (see the **Aggregation strategy** section below):
* **Extensive / count** variables (population totals, POI / building / facility counts) are **summed**.
* **Intensive** variables describing a condition *experienced by residents* (climate, air quality, NDVI, wealth index, internet speed, radiance, densities) are **population-weighted means**.
* **Land-area fractions** (land-cover %, hazard-zone %) are **area-weighted means**.

In [2]:
import json
import sys
from pathlib import Path

import numpy as np
import pandas as pd

REPO_ROOT = Path.cwd().resolve()
while not (REPO_ROOT / "data").exists() and REPO_ROOT.parent != REPO_ROOT:
    REPO_ROOT = REPO_ROOT.parent

DATA_DIR = REPO_ROOT / "data"
PROC_DIR = REPO_ROOT / "data" / "processed"
EDA_DIR = PROC_DIR / "eda"
HEALTH_DIR = PROC_DIR / "health"

TARGET_CITIES = sorted(json.loads((EDA_DIR / "target_adm3_pcodes.json").read_text()))
target_w = pd.read_csv(HEALTH_DIR / "all_cities_weekly_targets.csv", parse_dates=["date"])
WEEK_INDEX = sorted(target_w["date"].unique())
print("Target cities:", len(TARGET_CITIES), " weeks:", len(WEEK_INDEX))
print("Week range:", min(WEEK_INDEX), "->", max(WEEK_INDEX))

Target cities: 12  weeks: 888
Week range: 2005-12-26 00:00:00 -> 2022-12-26 00:00:00


In [3]:
def merge_multi_dfs(df_list, merge_on_cols=["date", "ADM4_PCODE"]):
    merged_df = pd.DataFrame()

    # Merge dataframes one by one
    for df in df_list:
        if merged_df.empty:
            merged_df = df
        else:
            # Merge on 'date' and 'adm4_pcode' columns
            merged_df = pd.merge(merged_df, df, on=merge_on_cols, how="outer")

    return merged_df

## 1. ADM4 -> ADM3 crosswalk + weights

Use `location.csv` for the ADM4 -> ADM3 hierarchy and build **two** weight columns:

* `_weight_area` -- `brgy_total_area` (km^2). Used for land-area fractions (land-cover %, hazard %), where the city value is "share of city *area*".
* `_weight_pop` -- a **static** per-barangay population (mean of WorldPop `pop_count_total` over all available years). Used for intensive variables that are *experienced by residents* (climate, air quality, wealth, connectivity), so a barangay with 50,000 people counts 50x a barangay with 1,000 when forming the city mean.

We deliberately use a *single, time-invariant* population weight so the weighting is identical across every feature block and year. This keeps features comparable over time and avoids circularity with the population feature we also emit. Barangays missing a population weight fall back to their area weight so they aren't silently dropped from population-weighted means.

In [5]:
loc = pd.read_csv(DATA_DIR / "location.csv")
loc = loc[loc["adm3_pcode"].isin(TARGET_CITIES)].copy()
loc = loc[["adm4_pcode", "adm3_pcode", "brgy_total_area"]].drop_duplicates("adm4_pcode")
# Rename to avoid a collision with feature tables that also publish a `brgy_total_area` column (e.g. brgy_geography.csv).
loc = loc.rename(columns={"brgy_total_area": "_weight_area"})

# Population weight for intensive ("experienced by residents") variables.
# Use a single static per-barangay population so the weighting is identical across
# every feature block and year (avoids the weight drifting year to year and avoids
# circularity with the population feature we also emit). Take the mean of WorldPop
# `pop_count_total` over all available years per barangay.
_pop = pd.read_csv(DATA_DIR / "worldpop_population.csv", usecols=["adm4_pcode", "pop_count_total"])
_pop = _pop.groupby("adm4_pcode", as_index=False)["pop_count_total"].mean()
_pop = _pop.rename(columns={"pop_count_total": "_weight_pop"})
loc = loc.merge(_pop, on="adm4_pcode", how="left")

# Barangays with no population weight fall back to area so they are not silently
# dropped from population-weighted means.
_n_missing_pop = int(loc["_weight_pop"].isna().sum())
loc["_weight_pop"] = loc["_weight_pop"].fillna(loc["_weight_area"])

print("ADM4s within target cities:", len(loc))
print("barangays missing pop weight (fell back to area):", _n_missing_pop)
loc

ADM4s within target cities: 879
barangays missing pop weight (fell back to area): 1


,adm4_pcode,adm3_pcode,_weight_area,_weight_pop
0,PH015518016,PH015518000,1.0216,809.965544
1,PH015518031,PH015518000,1.0440,5971.625488
2,PH015518022,PH015518000,3.2625,23480.196243
3,PH015518024,PH015518000,0.1814,1607.051589
4,PH034919017,PH034919000,6.3369,3030.916661
...,...,...,...,...
874,PH137401018,PH137401000,0.3856,8067.429455
875,PH137401022,PH137401000,1.0839,32220.777902
876,PH137503007,PH137503000,0.0581,2299.982178
877,PH137503014,PH137503000,5.7566,33596.140253


## 2. Shared helpers

Building blocks reused across all sources:
* `to_monday` -- snap a Series of timestamps to Monday-of-week.
* `adm4_weighted_to_adm3` -- **weighted mean** of numeric columns from ADM4 to ADM3 (per timestep group). `weight_col` selects population (`_weight_pop`, default) or area (`_weight_area`).
* `adm4_sum_to_adm3` -- **sum** of numeric columns from ADM4 to ADM3 (for extensive counts/totals).
* `classify_value_cols` / `adm4_to_adm3` -- type-aware aggregation: split columns into extensive (summed) vs intensive (weighted-mean'd) by name, then combine.
* `broadcast_to_weeks` -- expand a per-year (or per-month / static) ADM3 table to the full weekly grid.

In [7]:
def to_monday(s):
    return (s - pd.to_timedelta(s.dt.weekday, unit="D")).dt.normalize()


def adm4_weighted_to_adm3(df, value_cols, time_cols, weight_col="_weight_pop"):
    """Weighted mean of `value_cols` from ADM4 to ADM3, grouped by `time_cols`.

    `weight_col` selects the scheme: `_weight_pop` (population, the default -- for
    intensive variables experienced by residents) or `_weight_area` (barangay
    area -- for land-area fractions). A barangay only contributes weight where its
    value is non-null, so the result is a weighted mean over the barangays that
    actually carry data.
    """
    merged = df.merge(loc, on="adm4_pcode", how="inner")
    w = merged[weight_col].astype(float)
    keys = ["adm3_pcode"] + list(time_cols)
    out_chunks = []
    for col in value_cols:
        v = pd.to_numeric(merged[col], errors="coerce")
        wv = v * w
        tmp = merged[keys].copy()
        tmp["_wv"] = wv
        tmp["_w"] = np.where(v.notna(), w, np.nan)
        agg = tmp.groupby(keys, as_index=False).agg(_wv=("_wv", "sum"), _w=("_w", "sum"))
        agg[col] = agg["_wv"] / agg["_w"]
        out_chunks.append(agg[keys + [col]])
    res = out_chunks[0]
    for ch in out_chunks[1:]:
        res = res.merge(ch, on=keys, how="outer")
    return res


def adm4_sum_to_adm3(df, value_cols, time_cols):
    """Sum `value_cols` from ADM4 to ADM3, grouped by `time_cols`.

    For *extensive* quantities (counts, totals) the city value is the sum of its
    barangays. `min_count=1` keeps an all-missing city as NaN rather than a
    spurious 0.
    """
    merged = df.merge(loc, on="adm4_pcode", how="inner")
    keys = ["adm3_pcode"] + list(time_cols)
    num = merged[value_cols].apply(pd.to_numeric, errors="coerce")
    tmp = pd.concat([merged[keys], num], axis=1)
    return tmp.groupby(keys, as_index=False)[value_cols].sum(min_count=1)


# Substrings marking a column as *intensive* (a rate / mean / index / density /
# concentration -- must be averaged, never summed). Checked first.
INTENSIVE_MARKERS = (
    "mean", "median", "avg", "std", "stdev", "min", "max", "density",
    "rate", "ratio", "index", "pct", "percent", "norm", "ndvi", "rwi",
    "rad", "speed", "kbps", "lat_ms", "spi", "pnp", "dist",
)
# Substrings marking a column as *extensive* (a count / total to be summed).
COUNT_MARKERS = ("count", "total", "num_", "_num", "_sum", "n_tests", "n_devices")


def classify_value_cols(cols):
    """Split feature columns into (sum_cols, mean_cols) by name.

    Intensive markers win over count markers so e.g. `pop_count_mean` is averaged
    while `pop_count_total` is summed. Anything unmatched defaults to mean -- the
    safe side, since averaging a count is merely uninformative whereas summing a
    rate is outright wrong.
    """
    sum_cols, mean_cols = [], []
    for c in cols:
        cl = c.lower()
        if any(m in cl for m in INTENSIVE_MARKERS):
            mean_cols.append(c)
        elif any(m in cl for m in COUNT_MARKERS):
            sum_cols.append(c)
        else:
            mean_cols.append(c)
    return sum_cols, mean_cols


def adm4_to_adm3(df, value_cols, time_cols, weight_col="_weight_pop"):
    """Type-aware ADM4 -> ADM3 aggregation.

    Extensive columns are summed; intensive columns are weighted-mean'd with
    `weight_col`. Combines both into one frame keyed on `adm3_pcode` + `time_cols`.
    """
    sum_cols, mean_cols = classify_value_cols(value_cols)
    keys = ["adm3_pcode"] + list(time_cols)
    parts = []
    if sum_cols:
        parts.append(adm4_sum_to_adm3(df, sum_cols, time_cols))
    if mean_cols:
        parts.append(adm4_weighted_to_adm3(df, mean_cols, time_cols, weight_col=weight_col))
    res = parts[0]
    for p in parts[1:]:
        res = res.merge(p, on=keys, how="outer")
    return res


def broadcast_to_weeks(df, time_col, weeks_df):
    """Broadcast a per-period (annual / monthly / static) ADM3 frame to the full weekly grid."""
    if time_col is None:
        return weeks_df[["adm3_pcode", "date"]].merge(df, on="adm3_pcode", how="left")
    return weeks_df.merge(df, on=["adm3_pcode", time_col], how="left").drop(columns=["year", "month"], errors="ignore")


weeks_df = pd.DataFrame({"date": WEEK_INDEX})
weeks_df = weeks_df.merge(pd.DataFrame({"adm3_pcode": TARGET_CITIES}), how="cross")
weeks_df["year"] = weeks_df["date"].dt.year
weeks_df["month"] = weeks_df["date"].dt.to_period("M").dt.to_timestamp()
print("weeks_df:", weeks_df.shape)
weeks_df

weeks_df: (10656, 4)


,date,adm3_pcode,year,month
0,2005-12-26,PH015518000,2005,2005-12-01
1,2005-12-26,PH034919000,2005,2005-12-01
2,2005-12-26,PH050506000,2005,2005-12-01
3,2005-12-26,PH063022000,2005,2005-12-01
4,2005-12-26,PH072230000,2005,2005-12-01
...,...,...,...,...
10651,2022-12-26,PH104305000,2022,2022-12-01
10652,2022-12-26,PH112402000,2022,2022-12-01
10653,2022-12-26,PH137401000,2022,2022-12-01
10654,2022-12-26,PH137503000,2022,2022-12-01


### Aggregation strategy & justification

The target lives at the **city (ADM3)** level, but almost every feature is published at the **barangay (ADM4)** level, so each feature must be rolled up. The *correct* roll-up depends on what the variable physically represents.
| Variable type | Examples | Rule | Why |
|---|---|---|---|
| **Extensive (count / total)** | `pop_count_total`, OSM POI counts, building counts, DOH facility counts | **Sum** | The amount in a city *is* the sum of its barangays. Averaging would shrink a 50-barangay city to a per-barangay figure and destroy the population-size signal -- which matters most because `pop_count_total` is the natural **exposure/offset** for the count target. |
| **Intensive, resident-experienced** | temperature, rainfall, air quality, NDVI, RWI, internet speed, nightlight radiance, densities | **Population-weighted mean** | These describe a *condition*, not an amount, so they must be averaged. Weighting by **population** (not area) gives the value experienced by the *typical resident* -- the relevant quantity for a health/mortality model. A large, empty, low-NDVI barangay should not dominate the city's exposure just because it covers more land. |
| **Land-area fraction** | ESA WorldCover %, NOAH hazard-zone % | **Area-weighted mean** | These are shares *of area* by construction, so the city value is "share of city *area*", which is area-weighting by definition. |

**How columns are routed:**
* `classify_value_cols` splits columns by name: intensive markers (`mean`, `median`, `density`, `pct`, `rwi`, `rad`, ...) take precedence over count markers (`count`, `total`, `num_`, ...). Unmatched columns default to **mean** -- the safe side, since averaging a count is merely uninformative, whereas summing a rate is outright wrong.
* WorldCover/NOAH are forced through `_weight_area`; pure-count files (OSM POI, DOH POI) are summed wholesale; everything else uses the type-aware `adm4_to_adm3` with `_weight_pop`.

## 3. Daily climate -> weekly ADM3

Each daily climate file: filter ADM4 to target cities, snap date to Monday, average to weekly per ADM4, then **population-weighted** ADM4 -> ADM3. Climate / air-quality / NDVI are intensive variables experienced by residents, so population-weighting gives the exposure of the typical resident.

In [8]:
DAILY_CLIMATE_FILES = {
    "climate_air_quality.csv":          ["no2", "co", "so2", "o3", "pm10", "pm25"],
    "climate_atmosphere.csv":           ["tave", "tmin", "tmax", "heat_index", "pr", "wind_speed", "rh", "solar_rad", "uv_rad"],
    "climate_atmosphere_downscaled.csv":["tmin_downscaled", "tmax_downscaled", "tave_downscaled", "pr_downscaled"],
    "climate_land.csv":                 ["ndvi"],
}

valid_adm4 = set(loc["adm4_pcode"])

def process_daily_climate(path, value_cols, chunksize=500_000):
    weekly_chunks = []
    usecols = ["adm4_pcode", "date"] + value_cols
    for chunk in pd.read_csv(path, usecols=usecols, parse_dates=["date"], chunksize=chunksize):
        chunk = chunk[chunk["adm4_pcode"].isin(valid_adm4)]
        if len(chunk) == 0:
            continue
        chunk["date"] = to_monday(chunk["date"])
        # daily -> weekly mean per ADM4
        wk = chunk.groupby(["adm4_pcode", "date"], as_index=False)[value_cols].mean()
        weekly_chunks.append(wk)
    weekly = pd.concat(weekly_chunks, ignore_index=True)
    # Two chunks may share the same (adm4, week) at chunk boundaries; average again to merge.
    weekly = weekly.groupby(["adm4_pcode", "date"], as_index=False)[value_cols].mean()
    # ADM4 -> ADM3: intensive climate vars -> population-weighted mean.
    return adm4_weighted_to_adm3(weekly, value_cols, time_cols=["date"], weight_col="_weight_pop")

climate_dfs = []
for fname, vc in DAILY_CLIMATE_FILES.items():
    print("processing", fname, "...")
    out = process_daily_climate(DATA_DIR / fname, vc)
    print("  result shape:", out.shape)
    climate_dfs.append(out)

climate_daily = merge_multi_dfs(climate_dfs, merge_on_cols=["adm3_pcode", "date"])
print("daily-climate (weekly ADM3) shape:", climate_daily.shape)
climate_daily.head()

processing climate_air_quality.csv ...
  result shape: (12528, 8)
processing climate_atmosphere.csv ...
  result shape: (12528, 11)
processing climate_atmosphere_downscaled.csv ...
  result shape: (11484, 6)
processing climate_land.csv ...
  result shape: (12528, 3)
daily-climate (weekly ADM3) shape: (12528, 22)


,adm3_pcode,date,no2,co,so2,o3,pm10,pm25,tave,tmin,...,pr,wind_speed,rh,solar_rad,uv_rad,tmin_downscaled,tmax_downscaled,tave_downscaled,pr_downscaled,ndvi
0,PH015518000,2002-12-30,3.725000,0.109517,0.419204,37.814199,29.270000,19.957500,25.729025,22.928136,...,0.0,3.057035,63.866374,205.888073,23.872881,19.258135,31.685099,25.471766,0.0,0.427886
1,PH015518000,2003-01-06,4.098571,0.108165,0.678244,38.750786,28.667143,19.921429,26.037212,23.623501,...,0.0,2.325784,74.192900,181.045701,21.539570,20.617596,31.119362,25.868687,0.0,0.420592
2,PH015518000,2003-01-13,4.198571,0.103305,0.474141,36.687451,36.087143,25.148571,25.588060,22.782272,...,0.0,2.328100,67.363254,203.394129,23.257403,18.925423,31.406637,25.166285,0.0,0.416691
3,PH015518000,2003-01-20,4.285714,0.109816,0.662516,38.165940,41.322857,28.784286,25.208942,21.824315,...,0.0,2.856385,67.083258,222.668529,25.103357,16.962572,31.662580,24.313436,0.0,0.409485
4,PH015518000,2003-01-27,4.242857,0.110456,0.689096,38.502618,29.411429,20.497143,25.180520,21.960843,...,0.0,3.058354,67.892447,231.781125,26.169946,17.582632,31.579239,24.581640,0.0,0.388164


## 4. Monthly climate (indices) -> weekly ADM3 by month-broadcast

`pr_norm`, `spi3`, `spi6`, `pnp` are intensive climate indices, so they are **population-weighted** ADM4 -> ADM3 (per month), then broadcast to the weeks of that month.

In [9]:
indices_cols = ["pr_norm", "spi3", "spi6", "pnp"]
ci = pd.read_csv(
    DATA_DIR / "climate_indices.csv",
    usecols=["adm4_pcode", "date"] + indices_cols,
    parse_dates=["date"],
)
ci = ci[ci["adm4_pcode"].isin(valid_adm4)]
ci["month"] = ci["date"].dt.to_period("M").dt.to_timestamp()
ci = ci.drop(columns=["date"])
# Intensive climate indices -> population-weighted mean.
ci_adm3 = adm4_weighted_to_adm3(ci, indices_cols, time_cols=["month"], weight_col="_weight_pop")
climate_monthly = broadcast_to_weeks(ci_adm3, "month", weeks_df)
print("monthly-climate (weekly ADM3) shape:", climate_monthly.shape)
climate_monthly.head()

monthly-climate (weekly ADM3) shape: (10656, 6)


,date,adm3_pcode,pr_norm,spi3,spi6,pnp
0,2005-12-26,PH015518000,19.433906,0.141733,-1.474882,126.123792
1,2005-12-26,PH034919000,131.335876,0.193137,-0.354869,85.548646
2,2005-12-26,PH050506000,557.646279,1.033795,1.497457,181.599523
3,2005-12-26,PH063022000,106.771501,0.084154,0.284293,141.574035
4,2005-12-26,PH072230000,142.451580,0.486924,0.359561,162.627372


## 5. Annual ADM4 features -> weekly ADM3 (per-year broadcast)

For each file: read; pull `year` from `date`; aggregate ADM4 -> ADM3 per year; broadcast to weeks.

Aggregation strategy:
* **OSM POI** files are pure counts -> **summed** (the city has all the POIs of its barangays).
* **WorldPop / nightlights / Ookla / RWI** go through `adm4_to_adm3`: counts/totals (`pop_count_total`, `*_num_tests`, ...) are **summed**, while intensive fields (densities, radiance, speeds, wealth index) are **population-weighted means**.

In [10]:
ANNUAL_ADM4_FILES = {
    "worldpop_population.csv": [
        "pop_count_total", "pop_count_mean", "pop_count_median", "pop_count_stdev",
        "pop_count_min", "pop_count_max",
        "pop_density_mean", "pop_density_median", "pop_density_stdev",
        "pop_density_min", "pop_density_max",
    ],
    "nighttime_lights.csv": ["avg_rad_min", "avg_rad_max", "avg_rad_mean", "avg_rad_std", "avg_rad_median"],
    "ookla_internet_speed.csv": [
        "mobile_mean_avg_d_kbps_mean", "mobile_mean_avg_u_kbps_mean", "mobile_mean_avg_lat_ms_mean",
        "mobile_mean_num_tests_mean", "mobile_mean_num_devices_mean",
        "fixed_mean_avg_d_kbps_mean", "fixed_mean_avg_u_kbps_mean", "fixed_mean_avg_lat_ms_mean",
        "fixed_mean_num_tests_mean", "fixed_mean_num_devices_mean",
    ],
    "tm_relative_wealth_index.csv": ["rwi_max", "rwi_mean", "rwi_median", "rwi_min", "rwi_std"],
    "osm_poi_total.csv":     ["poi_count"],
    "osm_poi_amenity.csv":   None,  # detect numeric cols at runtime
    "osm_poi_health.csv":    None,
    "osm_poi_sanitation.csv":None,
    "osm_poi_water_body.csv":None,
}

# OSM POI tables are pure counts -> sum every numeric column.
COUNT_ONLY_FILES = {
    "osm_poi_total.csv", "osm_poi_amenity.csv", "osm_poi_health.csv",
    "osm_poi_sanitation.csv", "osm_poi_water_body.csv",
}

def numeric_value_cols(df, drop_cols=("uuid", "adm4_pcode", "adm3_pcode", "date", "freq", "geometry")):
    return [c for c in df.columns if c not in drop_cols and pd.api.types.is_numeric_dtype(df[c])]

annual_dfs = []
for fname, vc in ANNUAL_ADM4_FILES.items():
    path = DATA_DIR / fname
    if not path.exists():
        print("skip missing", fname)
        continue
    df = pd.read_csv(path, parse_dates=["date"])
    df = df[df["adm4_pcode"].isin(valid_adm4)].copy()
    if vc is None:
        vc = numeric_value_cols(df)
    df["year"] = df["date"].dt.year
    df = df[["adm4_pcode", "year"] + vc]
    if fname in COUNT_ONLY_FILES:
        adm3_yr = adm4_sum_to_adm3(df, vc, time_cols=["year"])
    else:
        # counts summed, intensive fields population-weighted
        adm3_yr = adm4_to_adm3(df, vc, time_cols=["year"], weight_col="_weight_pop")
    weekly = broadcast_to_weeks(adm3_yr, "year", weeks_df)
    print(f"{fname}: -> {weekly.shape}")
    annual_dfs.append(weekly)

annual_features = merge_multi_dfs(annual_dfs, merge_on_cols=["adm3_pcode", "date"])
print("annual_features shape:", annual_features.shape)

worldpop_population.csv: -> (10656, 13)
nighttime_lights.csv: -> (10656, 7)
ookla_internet_speed.csv: -> (10656, 12)
tm_relative_wealth_index.csv: -> (10656, 7)
osm_poi_total.csv: -> (10656, 3)
osm_poi_amenity.csv: -> (10656, 36)
osm_poi_health.csv: -> (10656, 14)
osm_poi_sanitation.csv: -> (10656, 23)
osm_poi_water_body.csv: -> (10656, 11)
annual_features shape: (10656, 110)


In [11]:
annual_features

,date,adm3_pcode,pop_count_total,pop_count_mean,pop_count_median,pop_count_stdev,pop_count_min,pop_count_max,pop_density_mean,pop_density_median,...,waste_transfer_station_nearest,osm_wetland_nearest,osm_reservoir_nearest,osm_water_nearest,osm_riverbank_nearest,osm_dock_nearest,osm_river_nearest,osm_stream_nearest,osm_canal_nearest,osm_drain_nearest
0,2005-12-26,PH015518000,132979.487671,53.450293,36.327333,40.686875,11.688490,179.041851,4002.718850,4124.554366,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,2006-01-02,PH015518000,137501.180176,55.249651,38.258575,39.368795,14.185419,170.857807,4122.622402,4222.567061,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,2006-01-09,PH015518000,137501.180176,55.249651,38.258575,39.368795,14.185419,170.857807,4122.622402,4222.567061,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,2006-01-16,PH015518000,137501.180176,55.249651,38.258575,39.368795,14.185419,170.857807,4122.622402,4222.567061,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,2006-01-23,PH015518000,137501.180176,55.249651,38.258575,39.368795,14.185419,170.857807,4122.622402,4222.567061,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
10651,2022-11-28,PH137603000,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,90000.0,12238.669415,36658.76298,0.0,6293.197145,90000.0,0.0,2655.02585,66512.579984,11125.826561
10652,2022-12-05,PH137603000,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,90000.0,12238.669415,36658.76298,0.0,6293.197145,90000.0,0.0,2655.02585,66512.579984,11125.826561
10653,2022-12-12,PH137603000,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,90000.0,12238.669415,36658.76298,0.0,6293.197145,90000.0,0.0,2655.02585,66512.579984,11125.826561
10654,2022-12-19,PH137603000,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,90000.0,12238.669415,36658.76298,0.0,6293.197145,90000.0,0.0,2655.02585,66512.579984,11125.826561


## 6. Static ADM4 features -> ADM3 (broadcast to all weeks)

Buildings, hazard %, worldcover %, brgy_geography, mapbox brgy isochrones, DOH POI inventory -> aggregated to ADM3 then held constant across weeks. Aggregation by file type:

* **`esa_worldcover`, `project_noah_hazards`** -- land-area fractions -> **area-weighted mean** (`_weight_area`).
* **`geoportal_doh_poi_health`** -- facility counts -> **summed**.
* **buildings, brgy_geography, mapbox brgy isochrones** -- mixed -> `adm4_to_adm3` (counts/totals summed, intensive fields population-weighted).

In [12]:
STATIC_ADM4_FILES = {
    "esa_worldcover.csv":   None,
    "project_noah_hazards.csv": None,
    "tm_open_buildings.csv":    None,
    "google_open_buildings.csv":None,
    "brgy_geography.csv":   None,
    "mapbox_health_facility_brgy_isochrones.csv": None,
    "geoportal_doh_poi_health.csv": None,
}

# Land-area fractions (% of barangay area) -> area-weighted mean.
AREA_WEIGHTED_FILES = {"esa_worldcover.csv", "project_noah_hazards.csv"}
# Pure-count inventories -> sum every numeric column.
COUNT_ONLY_STATIC = {"geoportal_doh_poi_health.csv"}

static_dfs = []
for fname in STATIC_ADM4_FILES:
    path = DATA_DIR / fname
    if not path.exists():
        print("skip missing", fname)
        continue
    df = pd.read_csv(path)
    df = df[df["adm4_pcode"].isin(valid_adm4)].copy()
    vc = numeric_value_cols(df)
    if not vc:
        print(f"  {fname}: no numeric columns, skipping")
        continue
    # collapse to one row per ADM4 (most static files already are)
    df_one = df.drop_duplicates("adm4_pcode")[["adm4_pcode"] + vc]
    df_one["_dummy"] = 1
    if fname in AREA_WEIGHTED_FILES:
        adm3 = adm4_weighted_to_adm3(df_one, vc, time_cols=["_dummy"], weight_col="_weight_area")
    elif fname in COUNT_ONLY_STATIC:
        adm3 = adm4_sum_to_adm3(df_one, vc, time_cols=["_dummy"])
    else:
        # mixed (buildings, geography, brgy isochrones): counts summed, intensive pop-weighted
        adm3 = adm4_to_adm3(df_one, vc, time_cols=["_dummy"], weight_col="_weight_pop")
    adm3 = adm3.drop(columns=["_dummy"])
    broadcast = broadcast_to_weeks(adm3, None, weeks_df)
    print(f"{fname}: -> {broadcast.shape}")
    static_dfs.append(broadcast)

static_features = merge_multi_dfs(static_dfs, merge_on_cols=["adm3_pcode", "date"])
print("static_features shape:", static_features.shape)

esa_worldcover.csv: -> (10656, 11)
project_noah_hazards.csv: -> (10656, 14)
tm_open_buildings.csv: -> (10656, 5)
google_open_buildings.csv: -> (10656, 10)
brgy_geography.csv: -> (10656, 5)
mapbox_health_facility_brgy_isochrones.csv: -> (10656, 20)
geoportal_doh_poi_health.csv: -> (10656, 19)
static_features shape: (10656, 72)


In [13]:
static_features

,adm3_pcode,date,pct_area_bare_sparse_vegetation,pct_area_builtup,pct_area_cropland,pct_area_grassland,pct_area_herbaceous_wetland,pct_area_mangroves,pct_area_permanent_water_bodies,pct_area_shrubland,...,doh_birthing_home_lying_in_clinic_count,doh_birthing_home_lying_in_clinic_nearest,doh_infirmary_count,doh_infirmary_nearest,doh_drug_abuse_treatment_rehabilitation_center_count,doh_drug_abuse_treatment_rehabilitation_center_nearest,doh_social_hygiene_clinic_count,doh_social_hygiene_clinic_nearest,doh_medical_clinic_count,doh_medical_clinic_nearest
0,PH015518000,2005-12-26,1.009267,24.107618,7.666529,12.096519,0.307293,0.0,42.176926,0.013294,...,3.0,32515.696352,0.0,310000.0,0.0,310000.0,0.0,310000.000000,0.0,310000.0
1,PH015518000,2006-01-02,1.009267,24.107618,7.666529,12.096519,0.307293,0.0,42.176926,0.013294,...,3.0,32515.696352,0.0,310000.0,0.0,310000.0,0.0,310000.000000,0.0,310000.0
2,PH015518000,2006-01-09,1.009267,24.107618,7.666529,12.096519,0.307293,0.0,42.176926,0.013294,...,3.0,32515.696352,0.0,310000.0,0.0,310000.0,0.0,310000.000000,0.0,310000.0
3,PH015518000,2006-01-16,1.009267,24.107618,7.666529,12.096519,0.307293,0.0,42.176926,0.013294,...,3.0,32515.696352,0.0,310000.0,0.0,310000.0,0.0,310000.000000,0.0,310000.0
4,PH015518000,2006-01-23,1.009267,24.107618,7.666529,12.096519,0.307293,0.0,42.176926,0.013294,...,3.0,32515.696352,0.0,310000.0,0.0,310000.0,0.0,310000.000000,0.0,310000.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
10651,PH137603000,2022-11-28,0.248068,77.269258,4.569619,5.509860,0.077616,0.0,0.137350,0.000000,...,9.0,2283.043983,0.0,90000.0,0.0,90000.0,1.0,19021.952112,0.0,90000.0
10652,PH137603000,2022-12-05,0.248068,77.269258,4.569619,5.509860,0.077616,0.0,0.137350,0.000000,...,9.0,2283.043983,0.0,90000.0,0.0,90000.0,1.0,19021.952112,0.0,90000.0
10653,PH137603000,2022-12-12,0.248068,77.269258,4.569619,5.509860,0.077616,0.0,0.137350,0.000000,...,9.0,2283.043983,0.0,90000.0,0.0,90000.0,1.0,19021.952112,0.0,90000.0
10654,PH137603000,2022-12-19,0.248068,77.269258,4.569619,5.509860,0.077616,0.0,0.137350,0.000000,...,9.0,2283.043983,0.0,90000.0,0.0,90000.0,1.0,19021.952112,0.0,90000.0


## 7. Already-ADM3 features (mapbox city isochrones)

In [14]:
mci = pd.read_csv(DATA_DIR / "mapbox_health_facility_city_isochrones.csv")
mci = mci[mci["adm3_pcode"].isin(TARGET_CITIES)].copy()
vc = numeric_value_cols(mci)
mci = mci[["adm3_pcode"] + vc].drop_duplicates("adm3_pcode")
city_iso_features = weeks_df[["adm3_pcode", "date"]].merge(mci, on="adm3_pcode", how="left")
print("city-iso features shape:", city_iso_features.shape)
city_iso_features.head()

city-iso features shape: (10656, 20)


,adm3_pcode,date,brgy_healthcenter_pop_reached_city_5min,brgy_healthcenter_pop_reached_city_15min,brgy_healthcenter_pop_reached_city_30min,brgy_healthcenter_pop_reached_city_pct_5min,brgy_healthcenter_pop_reached_city_pct_15min,brgy_healthcenter_pop_reached_city_pct_30min,hospital_pop_reached_city_5min,hospital_pop_reached_city_15min,hospital_pop_reached_city_30min,hospital_pop_reached_city_pct_5min,hospital_pop_reached_city_pct_15min,hospital_pop_reached_city_pct_30min,rhu_pop_reached_city_5min,rhu_pop_reached_city_15min,rhu_pop_reached_city_30min,rhu_pop_reached_city_pct_5min,rhu_pop_reached_city_pct_15min,rhu_pop_reached_city_pct_30min
0,PH015518000,2005-12-26,148946.05,206492.19,208321.78,71.50,99.12,100.0,121978.07,186700.85,203956.68,58.55,89.62,97.90,27039.46,143309.10,195188.75,12.98,68.79,93.70
1,PH034919000,2005-12-26,5025.45,27812.77,39672.71,9.04,50.06,71.4,0.00,0.00,0.00,0.00,0.00,0.00,3924.64,15011.39,34973.42,7.06,27.02,62.94
2,PH050506000,2005-12-26,135123.29,196102.84,198161.98,68.19,98.96,100.0,38672.05,149176.17,190932.68,19.52,75.28,96.35,0.00,0.00,0.00,0.00,0.00,0.00
3,PH063022000,2005-12-26,424837.19,424837.19,424837.19,100.00,100.00,100.0,180416.54,424837.19,424837.19,42.47,100.00,100.00,205658.72,422816.89,424837.19,48.41,99.52,100.00
4,PH072230000,2005-12-26,356164.91,356164.91,356164.91,100.00,100.00,100.0,207595.52,356164.91,356164.91,58.29,100.00,100.00,92598.40,327330.09,356164.91,26.00,91.90,100.00


## 8. Final merge

Start from `(adm3_pcode, date)` skeleton, then left-merge every block. Drop columns that ended up entirely NaN across the 12 target cities.

In [15]:
skeleton = weeks_df[["adm3_pcode", "date"]].copy()
blocks = [skeleton, climate_daily, climate_monthly, annual_features, static_features, city_iso_features]
features = merge_multi_dfs(blocks, merge_on_cols=["adm3_pcode", "date"])
print("merged features shape:", features.shape)

empty_cols = [c for c in features.columns if c not in ("adm3_pcode", "date") and features[c].isna().all()]
if empty_cols:
    print("dropping all-NaN cols:", empty_cols)
    features = features.drop(columns=empty_cols)

print("final feature table shape:", features.shape)
print("feature columns (first 20):", features.columns.tolist()[:20])

merged features shape: (12528, 222)
final feature table shape: (12528, 222)
feature columns (first 20): ['adm3_pcode', 'date', 'no2', 'co', 'so2', 'o3', 'pm10', 'pm25', 'tave', 'tmin', 'tmax', 'heat_index', 'pr', 'wind_speed', 'rh', 'solar_rad', 'uv_rad', 'tmin_downscaled', 'tmax_downscaled', 'tave_downscaled']


In [18]:
features.columns.tolist()

['adm3_pcode',
 'date',
 'no2',
 'co',
 'so2',
 'o3',
 'pm10',
 'pm25',
 'tave',
 'tmin',
 'tmax',
 'heat_index',
 'pr',
 'wind_speed',
 'rh',
 'solar_rad',
 'uv_rad',
 'tmin_downscaled',
 'tmax_downscaled',
 'tave_downscaled',
 'pr_downscaled',
 'ndvi',
 'pr_norm',
 'spi3',
 'spi6',
 'pnp',
 'pop_count_total',
 'pop_count_mean',
 'pop_count_median',
 'pop_count_stdev',
 'pop_count_min',
 'pop_count_max',
 'pop_density_mean',
 'pop_density_median',
 'pop_density_stdev',
 'pop_density_min',
 'pop_density_max',
 'avg_rad_min',
 'avg_rad_max',
 'avg_rad_mean',
 'avg_rad_std',
 'avg_rad_median',
 'mobile_mean_avg_d_kbps_mean',
 'mobile_mean_avg_u_kbps_mean',
 'mobile_mean_avg_lat_ms_mean',
 'mobile_mean_num_tests_mean',
 'mobile_mean_num_devices_mean',
 'fixed_mean_avg_d_kbps_mean',
 'fixed_mean_avg_u_kbps_mean',
 'fixed_mean_avg_lat_ms_mean',
 'fixed_mean_num_tests_mean',
 'fixed_mean_num_devices_mean',
 'rwi_max',
 'rwi_mean',
 'rwi_median',
 'rwi_min',
 'rwi_std',
 'poi_count',
 'atm_co

## 9. Write output

In [19]:
out_path = PROC_DIR / "all_cities_weekly_features.csv"
features.to_csv(out_path, index=False)
print("Wrote:", out_path)
print("Size (MB):", round(out_path.stat().st_size / 1e6, 1))

Wrote: /Users/adonaisray.maclang/Desktop/morbidity-mortality-modelling/data/processed/all_cities_weekly_features.csv
Size (MB): 26.5


In [21]:
# sparsity check for features
cols = [c for c in features.columns if c not in ("adm3_pcode", "date")]
n_rows = len(features)

missing = features[cols].isna().sum()
pct_missing = 100 * missing / n_rows
sparsity = pd.DataFrame({
    "missing": missing,
    "non_missing": n_rows - missing,
    "pct_missing": pct_missing
}).sort_values("pct_missing", ascending=False)

print("rows:", n_rows, "feature cols:", len(cols))
print("cols 100% missing:", (sparsity["pct_missing"] == 100).sum())
print("cols >90% missing:", (sparsity["pct_missing"] > 90).sum())
print("cols >50% missing:", (sparsity["pct_missing"] > 50).sum())
print("cols >10% missing:", (sparsity["pct_missing"] > 10).sum())
print("cols 0% missing:", (sparsity["pct_missing"] == 0).sum())

print("\nTop 20 most sparse features:")
print(sparsity.head(20))

print("\nTop 20 least sparse features:")
print(sparsity.tail(20))

rows: 12528 feature cols: 220
cols 100% missing: 0
cols >90% missing: 0
cols >50% missing: 93
cols >10% missing: 203
cols 0% missing: 16

Top 20 most sparse features:
                              missing  non_missing  pct_missing
fixed_mean_avg_u_kbps_mean      10032         2496    80.076628
mobile_mean_avg_d_kbps_mean     10032         2496    80.076628
mobile_mean_avg_u_kbps_mean     10032         2496    80.076628
mobile_mean_avg_lat_ms_mean     10032         2496    80.076628
mobile_mean_num_tests_mean      10032         2496    80.076628
mobile_mean_num_devices_mean    10032         2496    80.076628
fixed_mean_avg_d_kbps_mean      10032         2496    80.076628
fixed_mean_avg_lat_ms_mean      10032         2496    80.076628
fixed_mean_num_tests_mean       10032         2496    80.076628
fixed_mean_num_devices_mean     10032         2496    80.076628
rwi_mean                         8148         4380    65.038314
rwi_min                          8148         4380    65.038314
r